In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Charger les données
df = pd.read_csv('../data/processed/reclamations_clean.csv')
X = df['texte_clean']
y = df['categorie']

# Pipeline complète : vectoriseur + modèle
pipeline_finale = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95
    )),
    ('classifier', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        C=1.0
    ))
])

# Entraîner sur toutes les données (train + test confondus)
# car maintenant on veut le meilleur modèle final
print("Entraînement final sur toutes les données...")
pipeline_finale.fit(X, y)
print("✅ Entraînement terminé!")

# Sauvegarder
joblib.dump(pipeline_finale, '../models/classifier_pipeline.pkl')
print("✅ Modèle sauvegardé dans : models/classifier_pipeline.pkl")

# Sauvegarder aussi la liste des catégories
categories = list(pipeline_finale.classes_)
joblib.dump(categories, '../models/categories.pkl')
print(f"✅ Catégories sauvegardées : {categories}")

C:\Users\suki\AppData\Local\Temp\ipykernel_21976\2337855134.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Entraînement final sur toutes les données...
✅ Entraînement terminé!
✅ Modèle sauvegardé dans : models/classifier_pipeline.pkl
✅ Catégories sauvegardées : ['article_manquant', 'erreur_administrative', 'erreur_picking', 'mauvaise_qualite', 'probleme_transport', 'produit_casse', 'retard_livraison']


In [2]:
# Recharger comme si on était dans l'API
modele_charge = joblib.load('../models/classifier_pipeline.pkl')

# Tester
nouveaux_textes = [
    "Bonjour, j'ai commande un colis il y a deux semaines et je n'ai toujours rien recu",
    "Le produit que j'ai recu est completement casse, l'emballage etait abime",
    "Vous m'avez livre un mauvais produit, ce n'est pas du tout ce que j'ai commande",
    "Il manque plusieurs articles dans ma commande, c'est incomplet",
    "La qualite du produit laisse vraiment a desirer, tres decevant",
]

for texte in nouveaux_textes:
    pred = modele_charge.predict([texte])[0]
    probas = modele_charge.predict_proba([texte])[0]
    confiance = max(probas) * 100
    
    print(f"\n📝 Texte : {texte[:80]}...")
    print(f"   → Catégorie : {pred}")
    print(f"   → Confiance : {confiance:.1f}%")


📝 Texte : Bonjour, j'ai commande un colis il y a deux semaines et je n'ai toujours rien re...
   → Catégorie : retard_livraison
   → Confiance : 21.3%

📝 Texte : Le produit que j'ai recu est completement casse, l'emballage etait abime...
   → Catégorie : produit_casse
   → Confiance : 25.7%

📝 Texte : Vous m'avez livre un mauvais produit, ce n'est pas du tout ce que j'ai commande...
   → Catégorie : probleme_transport
   → Confiance : 37.0%

📝 Texte : Il manque plusieurs articles dans ma commande, c'est incomplet...
   → Catégorie : mauvaise_qualite
   → Confiance : 17.8%

📝 Texte : La qualite du produit laisse vraiment a desirer, tres decevant...
   → Catégorie : mauvaise_qualite
   → Confiance : 25.6%
